In [1]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import src.sqlqueries as sq
import sys
import os
import warnings
import utils.master as ma
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import sum as spark_sum, when
warnings.filterwarnings("ignore")


#Set the path for logging outputs
job_name = "sales_Aggregated_Metrics"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
ma.set_logging_path(data_working_path)

# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-bronze-ingestion")
    .getOrCreate()
)

ma.log("Spark Session initialized")

df = spark.read.parquet(
    "../data/cleansed/sales"
)
ma.log("Spark DataFrame created from silver - cleansed sales parquet files")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/02 19:51:29 WARN Utils: Your hostname, HP-665G11-353, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/02 19:51:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/02 19:51:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-02-02 19:51:49: Spark Session initialized


2026-02-02 19:51:58: Spark DataFrame created from silver - cleansed sales parquet files


**Gold Aggregations**

In [2]:
gold_df = (
    df.groupBy("Order_Year", "Order_Month")
      .agg(
          F.sum(F.col("Quantity_Ordered") * F.col("Price_Each")).alias("Total_Revenue"),
          F.sum("Quantity_Ordered").alias("Total_Units"),
          F.countDistinct("Order_ID").alias("Total_Orders")
      )
)


In [3]:
gold_df.write.mode("overwrite").parquet("../data/gold/sales_monthly")
